# 1.4 环境准备与第一个算子

## 概述

**本节要完成一件事：在950上运行一个TileLang加法算子，并确认结果正确。**

我们沿用上一节的分块例子，让两个各含1024项的数组相加。读完本节，你应该知道程序从哪里开始运行、计算由哪段代码描述，以及怎样判断运行成功。

<p class="course-note"><small>线上课程环境不提供950设备。本页保留真实运行输出，可直接对照阅读；自行运行请使用950环境中的终端，不要求安装Jupyter。</small></p>

## 1. 准备运行环境

请使用**已安装CANN的昇腾950环境**，并准备好Python、PyTorch、torch_npu和TileLang。TileLang的安装方法请参照[主仓README](https://github.com/tile-ai/tilelang#installation)。

在准备运行程序的终端中，先加载CANN环境：

```bash
source /usr/local/Ascend/cann/set_env.sh
```

<p class="course-note"><small>上面使用常见安装路径，请按本机位置调整。CANN通常已包含毕昇编译器。</small></p>

<details>
<summary>本页使用的软件版本</summary>

Python 3.12.9、CANN 9.1.0、PyTorch 2.11.0、torch_npu 2.11.0。

</details>

## 2. 先看清程序怎样完成加法

### 输入与预期结果

输入`a`从0开始递增，输入`b`的每一项都是1，数据类型都是`float32`。因此，**输出的每一项都应该比`a`中对应的值大1**。

本例取`N = 1024`、`TILE = 256`，也就是把1024项数据分成4块，每个任务处理256项。

### 程序的三个环节

先分清下面三个环节，再阅读完整代码：

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">环节</th>
<th style="text-align: left; vertical-align: top;">关键代码</th>
<th style="text-align: left; vertical-align: top;">在做什么</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;"><strong>描述计算</strong></td>
<td style="text-align: left; vertical-align: top;"><code>vector_add</code></td>
<td style="text-align: left; vertical-align: top;">写出每个任务怎样搬入数据、执行向量加法、写回结果。此时还没有对具体输入执行计算。</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><strong>编译算子</strong></td>
<td style="text-align: left; vertical-align: top;"><code>tilelang.compile(...)</code></td>
<td style="text-align: left; vertical-align: top;">将计算描述编译成面向950的程序，得到可以调用的<code>kernel</code>。</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><strong>调用并核对</strong></td>
<td style="text-align: left; vertical-align: top;"><code>out = kernel(a, b)</code></td>
<td style="text-align: left; vertical-align: top;">传入NPU上的<code>a</code>和<code>b</code>，得到输出<code>out</code>，再与PyTorch计算的<code>a + b</code>比较。</td>
</tr>
</tbody>
</table>

编译时，`target="ascend"`选择昇腾后端；`out_idx=-1`把最后一个参数`C`指定为输出，由调用时的返回值取得。

### 完整程序

代码中的三组注释与上表对应。先理解整体执行过程，向量循环、掩码和各个DSL接口的具体用法将在后续章节展开。

<p class="course-note"><small>程序文件：<code>src/01.04_first_operator.py</code>。首行<code>%%writefile</code>仅用于在Notebook中保存文件，不属于Python脚本；输出<code>Writing</code>或<code>Overwriting</code>表示保存完成。</small></p>

In [1]:
%%writefile src/01.04_first_operator.py
import torch
import torch_npu
import tilelang
import tilelang.language as T

tilelang.disable_cache()  # 本例每次运行都重新编译。

N = 1024
TILE = 256


# 1. 描述设备上怎样计算。
@T.prim_func
def vector_add(
    A: T.Buffer((N,), "float32"),
    B: T.Buffer((N,), "float32"),
    C: T.Buffer((N,), "float32"),
):
    with T.Kernel(N // TILE) as bx:
        a_ub = T.alloc_shared((TILE,), "float32")
        b_ub = T.alloc_shared((TILE,), "float32")
        c_ub = T.alloc_shared((TILE,), "float32")

        # 搬入当前数据块。
        begin = bx * TILE
        T.copy(A[begin : begin + TILE], a_ub)
        T.copy(B[begin : begin + TILE], b_ub)

        # 每轮加载一组数据，做向量加法，再存回UB。
        with T.SimdVF():
            mask = T.simd.pset(32)
            for r in range(TILE // 64):
                a_vec = T.simd.vld(a_ub[r * 64])
                b_vec = T.simd.vld(b_ub[r * 64])
                c_vec = T.simd.vadd(a_vec, b_vec, mask)
                T.simd.vsts(c_ub[r * 64], c_vec, mask)

        # 写回这一块结果。
        T.copy(c_ub, C[begin : begin + TILE])


# 2. 编译计算描述，得到可以调用的kernel。
kernel = tilelang.compile(vector_add, target="ascend", out_idx=-1)

# 3. 准备输入，调用kernel，再核对结果。
a = torch.arange(N, device="npu", dtype=torch.float32)
b = torch.ones(N, device="npu", dtype=torch.float32)
out = kernel(a, b)
torch.testing.assert_close(out, a + b, rtol=0, atol=0)
print("out[:8] =", out[:8].cpu().tolist())
print("Verification passed! (已比较全部1024项输出)")


Overwriting src/01.04_first_operator.py


<p class="course-note"><small><code>tilelang.disable_cache()</code>关闭当前Python进程的TileLang算子编译缓存，让本例每次运行都重新编译。</small></p>

## 3. 运行程序，核对结果

### 在终端运行

在课程仓库根目录进入本章目录，再运行脚本：

```bash
cd tutorials/tilelang_operator_development/01_overview
python src/01.04_first_operator.py
```

程序会先编译，再在设备上计算。终端中可能先出现编译日志，随后打印加法结果和校验信息。

### 对照输出检查

下面是本例的真实运行输出。**重点看加法结果`out[:8]`和校验信息`Verification passed!`。**

<p class="course-note"><small>如果使用Notebook运行：以本章目录为工作目录，先执行上方保存单元，再执行下方运行单元。它启动的是同一个Python脚本。</small></p>

In [2]:
# 可选：在Notebook中运行同一个脚本。
import os
import re
import subprocess
import sys

env = os.environ.copy()
env["PYTHONDONTWRITEBYTECODE"] = "1"
result = subprocess.run(
    [sys.executable, "src/01.04_first_operator.py"],
    env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
# 展示真实输出时隐去开发目录提示，其他信息与退出状态保留。
output = "".join(line for line in result.stdout.splitlines(keepends=True)
                 if "Loading tilelang libs from dev root:" not in line)
output = re.sub(r"/(?:home|root)/[^\s\"']+", "<本地路径>", output)
print(output, end="")
result.check_returncode()


[W918 18:04:35.826442453 FunctionLoader.cpp:48] Warning: LD_PRELOAD detected, FunctionLoader prefers RTLD_DEFAULT for symbol resolution. (function operator())
2026-09-18 18:04:41  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:131): TileLang begins to compile kernel `vector_add` with `out_idx=-1`
2026-09-18 18:04:53  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:139): TileLang completes to compile kernel `vector_add`
out[:8] = [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0]
Verification passed! (已比较全部1024项输出)


**加法结果：先看前八项是否符合预期。**

`out[:8]`只打印输出的前八项。输入`a`的前八项是0到7，加上`b`中对应的1后，输出应依次为1到8。

**校验信息：确认全部1024项都已通过比较。**

`torch.testing.assert_close`会把`out`与PyTorch计算的`a + b`逐项比较；只有比较通过，才会打印`Verification passed!`。因此，判断成功要看完整校验，不能只看前八项。

接下来可以完成[1.5的简短练习](01.05_chapter_practice.ipynb)，再进入[第2章](../02_basics/02.01_chapter_intro.ipynb)，学习怎样自己写出这些计算描述。